# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthGowda1718/ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression (with Random Forest as a comparison). Logistic Regression fits this lane well because the task is binary classification (needs_engagement_fix or not), the features are a mix of numeric and encoded categorical signals with mostly monotonic relationships to the label (e.g. worse ctr_gap → higher chance of needing a fix), and it produces interpretable coefficients — important since the content team needs to trust and explain why a page was flagged, not just get a black-box score. Random Forest is included as a comparison to see if nonlinear interactions (e.g. word_count only mattering at certain content_types) add real lift over the simpler model.

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

url = "https://raw.githubusercontent.com/jayanthGowda1718/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Rebuild label and features (same as w03/w04)
expected_ctr = df.groupby("position_tier")["ctr"].transform("median")
df["ctr_gap_ratio"] = df["ctr"] / expected_ctr
df["needs_engagement_fix"] = (df["ctr_gap_ratio"] < 0.6).astype(int)

numeric_features = ["search_volume", "competition", "cpc", "word_count", "char_count",
                     "impressions_90d", "sessions_90d", "users_90d",
                     "content_age_days", "avg_position"]
categorical_features = ["content_type", "main_intent", "freshness_tier", "word_count_tier"]

for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_features:
    df[col] = df[col].fillna("unknown")

X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features)
y = df["needs_engagement_fix"]

print(X.shape, y.value_counts())

(30000, 27) needs_engagement_fix
0    18417
1    11583
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: grouped by client_id, not a random row-level split. If the same client's pages appear in both train and test, the model can learn client-specific quirks (writing style, industry, template) rather than generalizable engagement patterns, which would make test performance look better than it really is on a brand-new client. A time-aware split isn't used here since this is a single 90-day snapshot, not a longitudinal feed — there's no meaningful "future" period to hold out yet.

In [8]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Client overlap between train/test:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

Train: (23837, 27) Test: (6163, 27)
Client overlap between train/test: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training Logistic Regression and Random Forest on the same client-grouped split, then comparing Precision@50 against the Week-4 baseline rule score, using the same metric and same test set so the comparison is fair.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).sort_values(ascending=False).head(k).index
    return y_true.iloc[top_k_idx].mean()

# Baseline: use ctr_gap_ratio itself as the "score" (lower gap ratio = higher priority)
baseline_scores = (1 - df.loc[X_test.index, "ctr_gap_ratio"].clip(upper=1)).reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)
baseline_p50 = precision_at_k(y_test_reset, baseline_scores, k=50)

# Logistic Regression with scaling
pipeline_log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(max_iter=1000))
])
pipeline_log_reg.fit(X_train, y_train)
log_reg_scores = pd.Series(pipeline_log_reg.predict_proba(X_test)[:, 1]).reset_index(drop=True)
log_reg_p50 = precision_at_k(y_test_reset, log_reg_scores, k=50)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_scores = pd.Series(rf.predict_proba(X_test)[:, 1]).reset_index(drop=True)
rf_p50 = precision_at_k(y_test_reset, rf_scores, k=50)

comparison = pd.DataFrame({
    "Method": ["Baseline rule", "Logistic Regression", "Random Forest"],
    "Precision@50": [baseline_p50, log_reg_p50, rf_p50]
})
comparison

,Method,Precision@50
0,Baseline rule,1.00
1,Logistic Regression,0.64
2,Random Forest,1.00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Looking at where the model disagrees with the baseline rule and with reality: the model tends to lean most heavily on avg_position and impressions_90d (the strongest signal from the Week-4 signal audit), which matches the earlier finding that position is the most reliable predictor of expected CTR. The model is most often wrong on pages with very low days_with_impressions — thin data produces an unstable ctr_gap_ratio, so both the baseline and the model can be confidently wrong in the same direction on these rows. This is a shared blind spot, not something the model fixes on its own, and suggests a data-quality filter (minimum days_with_impressions) should sit upstream of either method.

In [10]:
# Feature importance from the model (what it leans on)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

,0
impressions_90d,0.237072
avg_position,0.216882
sessions_90d,0.101629
users_90d,0.090404
content_age_days,0.070838
char_count,0.062679
word_count,0.062485
competition,0.032820
search_volume,0.032425
cpc,0.023007


In [11]:
# Look at cases where model and baseline disagree, and check days_with_impressions for those rows
model_preds = (log_reg_scores > 0.5).astype(int)
baseline_preds = (df.loc[X_test.index, "ctr_gap_ratio"].reset_index(drop=True) < 0.6).astype(int)

disagreement = df.loc[X_test.index].reset_index(drop=True)[model_preds != baseline_preds]
disagreement[["content_id", "days_with_impressions", "ctr_gap_ratio"]].head(10)

,content_id,days_with_impressions,ctr_gap_ratio
2,content_d4084a4bc775,88,0.1875
3,content_a5a2fbc76336,69,0.0000
8,content_033ae3e7aecf,18,0.0000
15,content_40cb4af260c0,6,0.0000
16,content_f0717373e86e,6,0.0000
17,content_d8a23b5e10c5,2,0.0000
19,content_dcebfd222b10,14,0.0000
20,content_caff51984338,37,0.0000
23,content_5607fec5d7db,14,0.0000
24,content_dea0d86223f3,21,0.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.